In [ ]:
import numpy as np
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.models.hbos import HBOS
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y

oof_ecod = np.zeros(len(X))
oof_copod = np.zeros(len(X))
oof_hbos = np.zeros(len(X))

synth_ecod = []
synth_copod = []
synth_hbos = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)

    
    ecod = ECOD(contamination=0.07)
    ecod.fit(X_tr_scaled)
    oof_ecod[val_idx] = ecod.decision_function(X_val_scaled)
    synth_ecod.extend(ecod.decision_function(X_synth_scaled))

    copod = COPOD(contamination=0.07)
    copod.fit(X_tr_scaled)
    oof_copod[val_idx] = copod.decision_function(X_val_scaled)
    synth_copod.extend(copod.decision_function(X_synth_scaled))

    hbos = HBOS(contamination=0.07)
    hbos.fit(X_tr_scaled)
    oof_hbos[val_idx] = hbos.decision_function(X_val_scaled)
    synth_hbos.extend(hbos.decision_function(X_synth_scaled))

print(f'ECOD OOF PR AUC: {average_precision_score(y, oof_ecod):.4f}')
print(f'COPOD OOF PR AUC: {average_precision_score(y, oof_copod):.4f}')
print(f'HBOS OOF PR AUC: {average_precision_score(y, oof_hbos):.4f}')

np.save('oof_copod.npy', oof_copod)
np.save('oof_ecod.py', oof_ecod)
np.save('oof_hbos.py', oof_hbos)

np.save('synth_ecod.npy', synth_ecod)
np.save('synth_copod.npy', synth_copod)
np.save('synth_hbos.npy', synth_hbos)

ECOD OOF PR AUC: 0.5684
COPOD OOF PR AUC: 0.7004
HBOS OOF PR AUC: 0.3241
